# Statistical analysis — main-study results

Implements the analysis plan in [docs/EXPERIMENT_PROTOCOL.md](../docs/EXPERIMENT_PROTOCOL.md) §7.

Reads `data/reports/<run_label>.csv` (long-format from `scripts/run_full_experiment.py` plus the appended `human_control` rows) and produces:

1. Per-metric Shapiro-Wilk + Levene checks
2. One-way ANOVA (or Kruskal-Wallis) across the 5 conditions, Bonferroni-corrected
3. Tukey HSD post-hoc on significant omnibus tests
4. Bootstrap 95% CIs per condition
5. Two-way ANOVA (condition × spec) for the interaction term
6. Forest plot per metric + violin plot per (condition × metric)

Runs unchanged on the real CSV once N≥5 per condition.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.formula.api as smf
import statsmodels.api as sm

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

RUN_LABEL = 'main_study_run_001'
CSV = Path('..') / 'data' / 'reports' / f'{RUN_LABEL}.csv'
ALPHA_PER_METRIC = 0.01        # Bonferroni: 0.05 / 5 metrics
BOOTSTRAP_RESAMPLES = 10_000

df = pd.read_csv(CSV) if CSV.exists() else pd.DataFrame()
print(f'Loaded {len(df)} rows from {CSV}')
df.head()

In [ ]:
# Quick sanity: do we have enough data?
if df.empty:
    raise SystemExit('No data yet. Run scripts/run_full_experiment.py first.')
counts = df.groupby(['condition','metric','spec_name']).size().unstack(fill_value=0)
print('Reps per (condition, metric, spec):')
counts

In [ ]:
def bootstrap_ci(values, n=BOOTSTRAP_RESAMPLES, ci=0.95):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return float(values.mean()) if len(values) else np.nan, np.nan, np.nan
    means = [np.random.choice(values, size=len(values), replace=True).mean() for _ in range(n)]
    lo, hi = np.quantile(means, [(1-ci)/2, 1-(1-ci)/2])
    return float(values.mean()), float(lo), float(hi)

summaries = []
for (metric, condition), sub in df.groupby(['metric','condition']):
    mean, lo, hi = bootstrap_ci(sub['value'])
    summaries.append({'metric':metric,'condition':condition,'n':len(sub),
                      'mean':mean,'ci_lo':lo,'ci_hi':hi})
summary = pd.DataFrame(summaries)
summary.pivot(index='condition', columns='metric', values='mean').round(3)

## Per-metric omnibus tests (Protocol §7.1)

In [ ]:
results = []
for metric in sorted(df['metric'].unique()):
    sub = df[df['metric'] == metric]
    groups = [g['value'].values for _, g in sub.groupby('condition')]
    if len(groups) < 2 or any(len(g) < 3 for g in groups):
        print(f'{metric:25s}: skipped (need >=3 obs per group)')
        continue
    normal = all(stats.shapiro(g).pvalue > 0.05 for g in groups)
    equal_var = stats.levene(*groups).pvalue > 0.05
    if normal and equal_var:
        test = 'ANOVA'; stat, p = stats.f_oneway(*groups)
    else:
        test = 'Kruskal-Wallis'; stat, p = stats.kruskal(*groups)
    sig = p < ALPHA_PER_METRIC
    grand_mean = sub['value'].mean()
    ss_between = sum(len(g)*(g.mean()-grand_mean)**2 for g in groups)
    ss_total = ((sub['value']-grand_mean)**2).sum()
    eta2 = ss_between/ss_total if ss_total else 0.0
    results.append({'metric':metric,'test':test,'stat':round(stat,3),
                    'p':round(p,5),'eta_sq':round(eta2,3),'significant':sig})
    print(f'{metric:25s} {test:14s} stat={stat:8.3f} p={p:.5f} η²={eta2:.3f} sig={sig}')
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Tukey HSD post-hoc on significant metrics
for _, row in results_df[results_df.get('significant', False) == True].iterrows():
    sub = df[df['metric'] == row['metric']]
    print(f"\n=== Tukey HSD: {row['metric']} ===")
    print(pairwise_tukeyhsd(sub['value'], sub['condition'], alpha=ALPHA_PER_METRIC))

## Condition × spec interaction (Protocol §7.2)

In [ ]:
for metric in sorted(df['metric'].unique()):
    sub = df[df['metric'] == metric].copy()
    if sub['spec_name'].nunique() < 2 or len(sub) < 10:
        continue
    model = smf.ols('value ~ C(condition) + C(spec_name) + C(condition):C(spec_name)', data=sub).fit()
    print(f'\n=== Two-way ANOVA: {metric} ===')
    print(sm.stats.anova_lm(model, typ=2).round(4))

## Forest plot per metric (mean ± 95% CI per condition)

In [ ]:
metrics = sorted(df['metric'].unique())
fig, axes = plt.subplots(len(metrics), 1, figsize=(8, 2.5*len(metrics)), squeeze=False)
for ax, metric in zip(axes.flat, metrics):
    rows = summary[summary['metric']==metric].sort_values('mean')
    y = np.arange(len(rows))
    ax.errorbar(rows['mean'], y, xerr=[rows['mean']-rows['ci_lo'], rows['ci_hi']-rows['mean']],
                fmt='o', color='#7c5cff', ecolor='#9b8aff', capsize=4)
    ax.set_yticks(y); ax.set_yticklabels(rows['condition'])
    ax.set_title(metric); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('forest_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Violin plots — distribution shape per (condition × metric)

In [ ]:
fig, axes = plt.subplots(len(metrics), 1, figsize=(8, 3*len(metrics)), squeeze=False)
for ax, metric in zip(axes.flat, metrics):
    sub = df[df['metric']==metric]
    conditions = sorted(sub['condition'].unique())
    data = [sub[sub['condition']==c]['value'].values for c in conditions]
    parts = ax.violinplot(data, showmeans=True, showmedians=True)
    ax.set_xticks(range(1, len(conditions)+1)); ax.set_xticklabels(conditions, rotation=20)
    ax.set_title(metric); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('violin_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Refusal / timeout rate (Protocol §8)

Computed from the provenance file, not the CSV (the CSV only contains successful scored runs).

In [ ]:
import json
prov_path = CSV.with_suffix('.provenance.json' if not str(CSV).endswith('.provenance.json') else '')
prov_path = CSV.parent / f'{RUN_LABEL}.provenance.json'
if prov_path.exists():
    prov = json.loads(prov_path.read_text())
    by_cond = pd.DataFrame(prov['statuses']).groupby('condition')['outcome'].value_counts().unstack(fill_value=0)
    by_cond['refusal_rate'] = 1 - by_cond.get('ok', 0) / by_cond.sum(axis=1)
    print('Refusal / failure rate per condition:')
    print(by_cond.round(3))
else:
    print('No provenance file yet.')

## Inter-rater reliability for the hallucination heuristic (Protocol §9)

Manual step: hand-label 30 random runs from `data/raw/`, save labels as
`data/labels/hallucination_handlabels.csv` with columns `run_id, n_hallucinated_handlabel`,
then run the cell below to compute Cohen's κ.

In [ ]:
from sklearn.metrics import cohen_kappa_score
labels = Path('..')/'data'/'labels'/'hallucination_handlabels.csv'
if labels.exists():
    hand = pd.read_csv(labels)
    auto = df[df['metric']=='hallucinations'].set_index('run_id')['value']
    merged = hand.merge(auto.rename('n_auto'), left_on='run_id', right_index=True, how='inner')
    if len(merged) >= 10:
        k = cohen_kappa_score(merged['n_hallucinated_handlabel'] > 0,
                              merged['n_auto'] > 0)
        print(f"Cohen's κ (any-hallucination presence) = {k:.3f}  on N={len(merged)}")
        print('Interpretation:  <0.4 poor · 0.4-0.6 moderate · 0.6-0.8 good · >0.8 very good')
    else:
        print(f'Have only {len(merged)} matched rows; need >= 10.')
else:
    print(f'No hand-labels yet. Expected at {labels}')